# 03 · 位置覆盖与逐案例配对评测
预计 40–50 分钟。闭环运行是在每次动作后接收新观测并产生下一动作。Notebook 02 已用两份学生模型各自运行标准5次、位置偏移5次；本节读取这些真实结果，不重新挑选测试集。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP)
from manipulation_lab.teaching import read_json, markdown_table, paired_cases
EXPERIMENT = stage_path(RUN, "experiment")
experiment = read_json(EXPERIMENT / "experiment.json")
assert experiment["complete"], "先完成 Notebook 02 的两组对照实验"
primary = experiment["variants"][experiment["primary_variant"]]
TRAINING = EXPERIMENT / primary["training"]
EVALUATION = EXPERIMENT / primary["evaluation"]


图中的训练点是160条训练示教**第一步**的易拉罐世界XY位置；标准与偏移点来自实际仿真日志的初始状态，连线连接同一个测试案例。不是全轨迹覆盖图。单位为米，世界Z向上。二维位置重叠或不重叠都不能证明高维输入分布是否相同。

In [ ]:
display(Image(filename=str(EVALUATION / "initial_positions.png")))
print("数据与划分：", read_json(TRAINING / "dataset_inspection.json")["dataset_sha256"])


两种条件使用同一组保留示教的初始机器人/物体状态及相同种子；偏移仅沿世界X增加4cm，并先检查仍在源容器内。**Position Shift 不等于严格 OOD（训练分布之外）测试**。稳定放置要求目标内、张开夹爪、无接触，并继续执行至少1秒满足位移和速度阈值；抬起不等于成功。

In [ ]:
for variant, paths in experiment["variants"].items():
    evaluation = EXPERIMENT / paths["evaluation"]
    results = read_json(evaluation / "summary.json")
    display(Markdown("### " + variant + " · LR=" + str(paths["learning_rate"])))
    print("checkpoint SHA256:", results["checkpoint_sha256"])
    display(Image(filename=str(evaluation / "success_rate_plot.png")))
    display(Markdown(markdown_table(paired_cases(results), [
        "source_episode", "seed", "standard_environment_success", "shifted_environment_success",
        "standard_stable_success", "shifted_stable_success", "standard_steps", "shifted_steps",
        "standard_error", "shifted_error"])))
    print("逐案例 CSV:", evaluation / "paired_results.csv")


In [ ]:
for filename in ["successful_rollout.mp4", "failed_rollout.mp4"]:
    path = EVALUATION / filename
    if path.exists():
        display(Markdown("**固定主模型 lr_a · " + filename + "**"))
        display(Video(str(path), embed=True))
    else:
        print(filename, "不存在：没有该类型回合，不能伪造视频。")


教师参考是另一个明确标记的模型。下格只查看教师预先准备的真实评测，不替换学生模型、不给学生测试重新选分。教师若未准备，本项明确显示未就绪。

In [ ]:
REFERENCE_ROOT = Path(os.environ.get("LAB_REFERENCE_ROOT", ROOT / "checkpoints"))
reference = REFERENCE_ROOT / "reference_evaluation/summary.json"
if reference.is_file():
    reference_results = read_json(reference)
    assert reference_results["policy_role"] == "instructor_reference"
    print("教师参考（独立模型）:", reference_results["conditions"])
else:
    print("教师参考未准备：请教师运行 scripts/prepare_reference.py；不影响已有学生结果。")
